In [ ]:
import pandas as pd
import seaborn as sns
import pingouin as pg
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from sklearn.cross_decomposition import CCA
from numpy.linalg import svd
from statsmodels.multivariate.cancorr import CanCorr
from sparsecca import pmd
from sklearn.model_selection import KFold
import itertools
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import miceforest as mf

### 1. Canonical corrleation analysis - 정준상관분석

두 변수 사이의 공통으로 작동하는 축을 찾아 두 집합의 연관 구조를 알고싶을 때 사용하는 분석 기법

In [67]:
# 데이터셋 불러오기 - taxi 데이터셋
df = sns.load_dataset('taxis')

In [68]:
# 데이터의 첫 5행 출력
df.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [69]:
X = df[['distance']]
Z = df[['fare', 'tip', 'tolls']]

scaler_X = StandardScaler()
scaler_Z = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
Z_scaled = scaler_Z.fit_transform(Z)

# Canonical correlation analysis 적용
cca = CCA(n_components=1)
U, V = cca.fit_transform(X_scaled, Z_scaled)

In [70]:
r_can, p_can = pearsonr(U[:, 0], V[:, 0])
print(f"Canonical correlation: {r_can:.6f}")
print(f"P-value: {p_can:.6f}")

Canonical correlation: 0.924995
P-value: 0.000000


In [71]:
# 최대 가능한 컴포넌트 수
max_k = min(X.shape[1], Z.shape[1], X.shape[0]-1)

# 모든 컴포넌트에 대해 CCA 실행
cca = CCA(n_components=max_k)
U, V = cca.fit_transform(X, Z)

# 각 컴포넌트의 canonical correlation 계산
cors = []
for i in range(max_k):
    r = np.corrcoef(U[:, i], V[:, i])[0, 1]
    cors.append(r)
    print(f"Component {i+1}: correlation = {r:.4f}")

# 급격히 떨어지는 지점 찾기 (예: 0.1 이상 차이)
best_n = 1
for i in range(len(cors)-1):
    if cors[i] - cors[i+1] > 0.1:  # 임계값 조정 가능
        best_n = i + 1
        break
else:
    best_n = len([c for c in cors if c > 0.1])  # 0.1 이상인 컴포넌트 수

print(f"\nRecommended n_components: {best_n}")

Component 1: correlation = 0.9250

Recommended n_components: 1


In [72]:
# Permutation Test - 결과가 우연하게 나오는지 확인하는 검증
# p-value가 0.05보다 작으면 유의미하다고 판단
def cca_statistic(X_scaled, Z_scaled):
    cca = CCA(n_components=1)
    U, V = cca.fit_transform(X_scaled, Z_scaled)
    return pearsonr(U[:, 0], V[:, 0])[0]  # correlation만 반환

# Observed correlation
r_obs = cca_statistic(X_scaled, Z_scaled)
print(f"\n=== Permutation Test ===")
print(f"Observed correlation: {r_obs:.6f}")

# Permutation test
B = 1000  # permutation 횟수
rng = np.random.default_rng(0)
r_perm = []

for i in range(B):
    # X의 행을 무작위로 섞음
    idx = rng.permutation(X_scaled.shape[0])
    X_perm = X_scaled[idx, :]
    r_perm.append(cca_statistic(X_perm, Z_scaled))
r_perm = np.array(r_perm)

# p-value 계산
p_perm = np.mean(np.abs(r_perm) >= np.abs(r_obs))

print(f"Permutation p-value: {p_perm:.4f}")

# 변수 중요도
X_cols = ['distance']
wX = pd.Series(cca.x_weights_[:, 0], index=X_cols).sort_values(key=np.abs, ascending=False)
print("\nX 변수 중요도")
print(wX)

Z_cols = ['fare', 'tip', 'tolls']
wZ = pd.Series(cca.y_weights_[:, 0], index=Z_cols).sort_values(key=np.abs, ascending=False)
print("\nZ 변수 중요도")
print(wZ)


=== Permutation Test ===
Observed correlation: 0.924995
Permutation p-value: 0.0000

X 변수 중요도
distance    1.0
dtype: float64

Z 변수 중요도
fare     0.989890
tolls    0.140830
tip     -0.016851
dtype: float64


결과 : canonical correlation 값이 0.924995로 높은 상관관계를 보인다는 것을 알 수 있으며, fare이 distance와 가장 강하게 움직임

### 2. Sparse CCA - 희소 정준상관분석

두 변수 집합의 공통 퍁천을 찾되, 변수 수가 많을 때 핵심 변수만 선택해 해석함

주로 변수가 많은 경우에 사용

In [73]:
from sklearn.datasets import load_diabetes
X_all, y = load_diabetes(return_X_y=True, as_frame=True)

X = X_all[['bmi','bp','s1','s2','s3']]
Z = X_all[['s4','s5','s6','age','sex']]

# z-score 표준화 
X = ((X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)).to_numpy()
Z = ((Z - Z.mean(axis=0)) / (Z.std(axis=0) + 1e-12)).to_numpy()

In [74]:
# 최대 가능한 컴포넌트 수
max_k = min(X.shape[1], Z.shape[1], X.shape[0]-1)

# 모든 컴포넌트에 대해 CCA 실행
cca = CCA(n_components=max_k)
U, V = cca.fit_transform(X, Z)

# 각 컴포넌트의 canonical correlation 계산
cors = []
for i in range(max_k):
    r = np.corrcoef(U[:, i], V[:, i])[0, 1]
    cors.append(r)
    print(f"Component {i+1}: correlation = {r:.4f}")

# 급격히 떨어지는 지점 찾기 (예: 0.1 이상 차이)
best_n = 1
for i in range(len(cors)-1):
    if cors[i] - cors[i+1] > 0.1:  # 임계값 조정 가능
        best_n = i + 1
        break
else:
    best_n = len([c for c in cors if c > 0.1])  # 0.1 이상인 컴포넌트 수

print(f"\nRecommended n_components: {best_n}")

Component 1: correlation = 0.9692
Component 2: correlation = 0.8338
Component 3: correlation = 0.3727
Component 4: correlation = 0.2804
Component 5: correlation = 0.1243

Recommended n_components: 1


In [75]:
# 5번의 cross validation으로 penaltyu, penaltyv 조합 평가
def evaluate_penalty_combination(X, Z, penaltyu, penaltyv, cv=5):
    kf = KFold(n_splits=cv, shuffle=True, random_state=0)
    correlations = []
    
    # train, val 나누기
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        Z_train, Z_val = Z[train_idx], Z[val_idx]
        
        try:
            # Train
            U, V, D = pmd(X_train.T @ Z_train, K=1, 
                         penaltyu=penaltyu, penaltyv=penaltyv, standardize=False)
            x_weights = U[:, 0]
            z_weights = V[:, 0]
            
            # Validate
            corr = np.corrcoef(np.dot(x_weights, X_val.T), 
                              np.dot(z_weights, Z_val.T))[0, 1]
            if not np.isnan(corr):
                correlations.append(corr)
        except:
            continue
    
    if len(correlations) == 0:
        return np.nan, np.nan
    
    return np.mean(correlations), np.std(correlations)

from scipy.optimize import minimize

# mean_corr을 최대화하는 것이 목적
def objective(params):
    penaltyu, penaltyv = params
    mean_corr, _ = evaluate_penalty_combination(X, Z, penaltyu, penaltyv, cv=3)
    return -mean_corr  # 최소화하려면 음수

# 초기값은 0.5로 설정
initial_params = [0.5, 0.5]
bounds = [(0.1, 1.0), (0.1, 1.0)]  # penalty 범위

result = minimize(objective, initial_params, method='L-BFGS-B', bounds=bounds)
best_penaltyu, best_penaltyv = result.x

print(f"최적 penaltyu: {best_penaltyu:.3f}")
print(f"최적 penaltyv: {best_penaltyv:.3f}")

최적 penaltyu: 1.000
최적 penaltyv: 0.356


In [76]:
def sparse_cca_statistic(X, Z, penaltyu=1, penaltyv=0.356):
    U, V, D = pmd(X.T @ Z, K=1, penaltyu=penaltyu, penaltyv=penaltyv, standardize=False)
    x_weights = U[:, 0]
    z_weights = V[:, 0]
    corrcoef = np.corrcoef(np.dot(x_weights, X.T), np.dot(z_weights, Z.T))[0, 1]
    return corrcoef

# Observed correlation
r_obs = sparse_cca_statistic(X, Z, penaltyu=1, penaltyv=0.356)
print(f"Observed correlation: {r_obs:.6f}")

Observed correlation: 0.863675


X와 Z사이의 공통 축이 강하게 존재함 (r=0.863675)

In [77]:
# Permutation test
B = 1000 
rng = np.random.default_rng(0)
r_perm = []

for i in range(B):
    # X의 행(row)을 무작위로 섞음
    idx = rng.permutation(X.shape[0])
    X_perm = X[idx, :]
    r_perm.append(sparse_cca_statistic(X_perm, Z, penaltyu=1, penaltyv=0.356))

r_perm = np.array(r_perm)

# p-value 계산: observed correlation 이상의 값이 나온 비율
p_perm = np.mean(np.abs(r_perm) >= np.abs(r_obs))

print(f"Permutation p-value: {p_perm:.4f}")
print(f"95% percentile of permuted correlations: {np.percentile(r_perm, 95):.6f}")
print(f"99% percentile of permuted correlations: {np.percentile(r_perm, 99):.6f}")

Permutation p-value: 0.0000
95% percentile of permuted correlations: 0.157212
99% percentile of permuted correlations: 0.178398


관측 상관(0.864)은 permutation 상위 99% 컷(0.178)보다 훨씬 커 우연으로 보기 어려움

In [78]:
# 가중치 확인 - x, z 변수 중요도
U, V, D = pmd(X.T @ Z, K=1, penaltyu=1, penaltyv=0.356, standardize=False)
x_weights = U[:, 0]
z_weights = V[:, 0]

# X 변수 이름
X_cols =['bmi','bp','s1','s2','s3']
# Z 변수 이름
Z_cols = ['s4','s5','s6','age','sex']


wX = pd.Series(x_weights, index=X_cols)
wX_selected = wX[np.abs(wX) > 1e-12].sort_values(key=np.abs, ascending=False)

print("\n선택된 X 변수 가중치")
print(wX_selected)

wZ = pd.Series(z_weights, index=Z_cols)
wZ_selected = wZ[np.abs(wZ) > 1e-12].sort_values(key=np.abs, ascending=False)

print("\n선택된 Z 변수 가중치")
print(wZ_selected)


선택된 X 변수 가중치
s3     0.600512
s2    -0.536536
s1    -0.440901
bmi   -0.336491
bp    -0.209511
dtype: float64

선택된 Z 변수 가중치
s4   -1.0
dtype: float64


결과 : X 에서 선택된 변수는 s3, s2, s1, bmi, bp이고 Z에서는 s4만 선택되어 s4 하나만 써도 X쪽과 공통 축을 충분히 잘 만들 수 있음


### 3. Partial correlation analysis - 부분 상관관계 분석

두 변수 사이의 관계를 볼 때, 영향을 줄 수 잇는 다른 변수를 통제한 뒤 순수한 상관만 확인하고 싶을 때 사용

In [79]:
# distance, tolls를 통제변수를 둔 후에 fare과 tip의 부분 상관관계 분석
columns = ['fare']
for c in columns:
   print('result of tip & ', c, '\n',pg.partial_corr(data=df, x='tip', y= c , covar=['distance', 'tolls'], method='pearson'))

result of tip &  fare 
             n        r         CI95%         p-val
pearson  6433  0.19562  [0.17, 0.22]  1.710882e-56


### 4. Partial Least Squares Regression - 부분최소제곱 회귀

설명변수(X)가 많고 서로 상관이 큰 상황에서, X를 몇 개의 성분으로 압축하면서 목표변수(y)를 잘 예측하는 회귀모형을 만들고 싶을 때 사용합니다

In [80]:
X = df[['distance', 'tolls', 'tip']]
y = df[['fare']]

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

In [81]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

max_comp = min(X_scaled.shape[1], X_scaled.shape[0]-1)  
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_list = []
for n in range(1, max_comp+1):
    rmses = []
    for tr, te in kf.split(X_scaled):
        pls_cv = PLSRegression(n_components=n)
        pls_cv.fit(X_scaled[tr], y_scaled[tr])
        pred = pls_cv.predict(X_scaled[te])
        rmses.append(np.sqrt(mean_squared_error(y_scaled[te], pred)))
    rmse_list.append(np.mean(rmses))

best_n = int(np.argmin(rmse_list) + 1)
print("best n_components =", best_n)

best n_components = 3


In [82]:
pls = PLSRegression(n_components=3)
pls.fit(X_scaled, y_scaled)

# 예측
y_pred = pls.predict(X_scaled)

# Correlation 계산
r, p = pearsonr(y_scaled[:, 0], y_pred[:, 0])
print(f"PLSR : Component과 fare의 correlation: {r:.4f} (p={p:.4f})")

# 변수 중요도=> Loadings
X_cols = ['distance', 'tolls', 'tip']
loadings = pd.Series(pls.x_loadings_[:, 0], index=X_cols).sort_values(key=abs, ascending=False)

print("\n=== PLSR Loadings  ===")
print(loadings)

# Weights 확인 
weights = pd.Series(pls.x_weights_[:, 0], index=X_cols).sort_values(key=abs, ascending=False)
print("\n=== PLSR Weights ===")
print(weights)

PLSR : Component과 fare의 correlation: 0.9238 (p=0.0000)

=== PLSR Loadings  ===
distance    0.653595
tolls       0.596973
tip         0.494825
dtype: float64

=== PLSR Weights ===
distance    0.762376
tolls       0.504855
tip         0.404851
dtype: float64


distance, tolls, tip을 결합한 plsr모델은 훈련 데이터에서 fare을 잘 맞춤(r=0.9238)

In [83]:
from sklearn.model_selection import permutation_test_score

pls = PLSRegression(n_components=1)
score, perm_scores, p_perm = permutation_test_score(
    pls,
    X_scaled,
    y_scaled.ravel(),
    scoring="r2",
    n_permutations=1000,
    random_state=42
)

print(f"CV R2: {score:.4f}")
print(f"Permutation test p-value: {p_perm:.4f}")

CV R2: 0.7465
Permutation test p-value: 0.0010


결과: 교차검증 cv를 통해 74.65% 설명한다는 것을 알 수 있으며, permutation test 결과로도 통계적으로 유의미한 예측 성능을 보임

### 5. 결측치 처리 방법 비교

먼저 데이터셋을 실제 데이터와 유사하게 변형

In [81]:
df = sns.load_dataset('taxis')
df2 = df.sample(n=44, random_state = 42).reset_index(drop=True)

In [82]:
# 원본 결과
c = 'fare'
pg.partial_corr(data=df2, x='distance', y= c , covar=['tolls', 'tip'])

C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:866: RuntimeWarning: divide by zero encountered in divide
  D = np.diag(np.sqrt(1 / Vi_diag))
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:867: RuntimeWarning: invalid value encountered in matmul
  pcor = -1 * (D @ Vi @ D)  # Partial correlation matrix


,n,r,CI95%,p-val
pearson,44,0.920132,"[0.86, 0.96]",7.023564e-18


In [83]:
cols = ['distance', 'passengers'] 
for col in cols:
    idx = np.random.choice(df2.index, size=10, replace=False)
    df2.loc[idx, col] = np.nan

In [84]:
df2.isna().sum()

pickup              0
dropoff             0
passengers         10
distance           10
fare                0
tip                 0
tolls               0
total               0
color               0
payment             0
pickup_zone         0
dropoff_zone        0
pickup_borough      0
dropoff_borough     0
dtype: int64

In [85]:
# 결측치를 추가하면 결과가 좀 이상해짐
c = 'fare'
pg.partial_corr(data=df2, x='distance', y= c , covar=['tolls', 'tip'])

C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:866: RuntimeWarning: divide by zero encountered in divide
  D = np.diag(np.sqrt(1 / Vi_diag))
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:867: RuntimeWarning: invalid value encountered in matmul
  pcor = -1 * (D @ Vi @ D)  # Partial correlation matrix


,n,r,CI95%,p-val
pearson,34,0.93442,"[0.87, 0.97]",5.459156e-15


#### (1) 다중대체

In [89]:
# 최적화된 결측치 처리 방법

impute_cols = ['distance',  'passengers']
keep_cols_mice = ['fare', 'tip', 'tolls']

# 사용 가능한 컬럼만 선택
available_cols_mice = [col for col in keep_cols_mice if col in df2.columns]
use_cols_mice = available_cols_mice + impute_cols
df_sub_mice = df2[use_cols_mice].copy()

kernel_mice = mf.ImputationKernel(
    data=df_sub_mice,
    num_datasets=5,
    save_all_iterations_data=True,
    random_state=41
)
kernel_mice.mice(iterations=10)

df_mice = df2.copy().reset_index(drop=True)
df_mice[impute_cols] = kernel_mice.complete_data(dataset=0)[impute_cols]


In [87]:
# 5개의 mice 결과 dataset의 평균 - 새로운 데이터셋 df_mean으로 저장
m = kernel_mice.num_datasets
stack = np.stack([
    kernel_mice.complete_data(dataset=i)[impute_cols].to_numpy()
    for i in range(m)
], axis=0)

df_mean = df2.copy()
df_mean[impute_cols] = stack.mean(axis=0)
df_mean['distance'].head()

0    1.300
1    1.488
2    2.390
3    1.900
4    2.400
Name: distance, dtype: float64

In [88]:
c = 'fare'
pg.partial_corr(data=df_mean, x='distance', y= c , covar=['tolls'])

C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:866: RuntimeWarning: divide by zero encountered in divide
  D = np.diag(np.sqrt(1 / Vi_diag))
C:\Users\nva_kist\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pingouin\correlation.py:867: RuntimeWarning: invalid value encountered in matmul
  pcor = -1 * (D @ Vi @ D)  # Partial correlation matrix


,n,r,CI95%,p-val
pearson,44,0.948949,"[0.91, 0.97]",3.711066e-22


#### (2) Low-rank matrix completion

#### (3) 삭제